# Fish Feeder Research Analysis

Load one experiment CSV from `../data/raw`, inspect the telemetry stream, and visualize the feeder response.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

DATA_DIR = Path('..') / 'data' / 'raw'
csv_files = sorted(DATA_DIR.glob('*.csv'))
csv_files[-5:]

In [ ]:
csv_path = csv_files[-1]
df = pd.read_csv(csv_path)
df.head()

In [ ]:
numeric_cols = ['temp', 'biomass', 'fuzzy_output_g', 'feed_estimate_g', 'distance_mm', 'duration_s']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df[['temp', 'biomass', 'fuzzy_output_g']].describe()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['temp'], df['fuzzy_output_g'], alpha=0.7)
plt.xlabel('Temperature (C)')
plt.ylabel('Feed Output (g)')
plt.title('Temperature vs Feed Output')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['biomass'], df['fuzzy_output_g'], alpha=0.7, color='tab:green')
plt.xlabel('Biomass (g)')
plt.ylabel('Feed Output (g)')
plt.title('Biomass vs Feed Output')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
surface = (
    df.dropna(subset=['temp', 'biomass', 'fuzzy_output_g'])
      .groupby(['temp', 'biomass'], as_index=False)['fuzzy_output_g']
      .mean()
      .pivot(index='biomass', columns='temp', values='fuzzy_output_g')
      .sort_index()
)

fig = go.Figure(
    data=[
        go.Surface(
            z=surface.values,
            x=surface.columns.values,
            y=surface.index.values,
            colorbar={'title': 'Output (g)'}
        )
    ]
)
fig.update_layout(
    title='Temperature vs Biomass vs Feed Output',
    scene={'xaxis_title': 'Temperature (C)', 'yaxis_title': 'Biomass (g)', 'zaxis_title': 'Feed Output (g)'}
)
fig.show()

In [ ]:
if 'real_weight_g' in df.columns:
    error = df['real_weight_g'] - df['fuzzy_output_g']
    mae = error.abs().mean()
    rmse = np.sqrt((error ** 2).mean())
    print({'mae_g': float(mae), 'rmse_g': float(rmse)})
else:
    print('Add a real_weight_g column to run error analysis.')